In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def generate_sales_data(samples=365):
    """Generates synthetic daily retail sales data for machine learning forecasting."""
    np.random.seed(42)
    
    # Features
    day_of_week = np.random.choice(range(1, 8), samples)          # 1 = Monday, 7 = Sunday
    is_promo = np.random.choice([0, 1], samples, p=[0.7, 0.3])     # 30% days have promo
    store_footfall = np.random.uniform(200, 2000, samples)          # Visitors
    avg_discount_pct = np.random.uniform(0, 40, samples)           # % Discount
    seasonality_factor = np.random.uniform(0.8, 1.5, samples)      # Peak demand factor
    
    # Non-linear relationship for Realistic Sales
    base_sales = 500
    weekend_boost = np.where(day_of_week >= 6, 250, 0)
    promo_boost = is_promo * 400
    
    noise = np.random.normal(0, 150, samples)
    
    units_sold = (
        (base_sales + weekend_boost + promo_boost) * seasonality_factor
        + (0.35 * store_footfall)
        + (12 * avg_discount_pct)
        + noise
    )
    
    df = pd.DataFrame({
        'Day_of_Week': day_of_week,
        'Is_Promo': is_promo,
        'Store_Footfall': store_footfall,
        'Avg_Discount_Pct': avg_discount_pct,
        'Seasonality_Factor': seasonality_factor,
        'Units_Sold': np.maximum(50, units_sold.round())
    })
    return df

def train_sales_forecasting_model(df):
    """Trains a Random Forest Regressor and evaluates forecasting metrics."""
    features = ['Day_of_Week', 'Is_Promo', 'Store_Footfall', 'Avg_Discount_Pct', 'Seasonality_Factor']
    X = df[features]
    y = df['Units_Sold']
    
    # Train/Test Split (80% historical training, 20% future testing)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Instantiate Random Forest Regressor
    rf_model = RandomForestRegressor(
        n_estimators=100, 
        max_depth=12, 
        random_state=42
    )
    rf_model.fit(X_train, y_train)
    
    # Predictions & Metrics
    y_pred = rf_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    return rf_model, r2, mae, rmse

def forecast_future_sales(model, scenario):
    """Generates predicted units sold for a specific future sales scenario."""
    input_df = pd.DataFrame([scenario])
    predicted_sales = model.predict(input_df)[0]
    return int(round(predicted_sales))

# --- Main Execution Block ---
if __name__ == "__main__":
    # 1. Dataset Generation
    data = generate_sales_data(samples=365)
    print("--- Historical Daily Sales Dataset (First 5 Rows) ---")
    print(data.head().round(2).to_string(index=False))
    print("\n" + "="*60 + "\n")
    
    # 2. Model Training & Evaluation
    model, r2, mae, rmse = train_sales_forecasting_model(data)
    
    print("--- Model Performance Metrics ---")
    print(f"R² Score (Accuracy Metric)     : {r2:.4f}")
    print(f"Mean Absolute Error (MAE)      : {mae:.2f} units")
    print(f"Root Mean Squared Error (RMSE) : {rmse:.2f} units")
    print("\n" + "="*60 + "\n")
    
    # 3. Feature Importances
    print("--- Key Sales Drivers (Feature Importance) ---")
    features = ['Day_of_Week', 'Is_Promo', 'Store_Footfall', 'Avg_Discount_Pct', 'Seasonality_Factor']
    importances = model.feature_importances_
    
    for feat, imp in zip(features, importances):
        print(f"Feature [{feat:20s}] Impact: {imp * 100:.2f}%")
    print("\n" + "="*60 + "\n")
    
    # 4. Sales Forecast Scenario Execution
    upcoming_day = {
        'Day_of_Week': 6,             # Saturday
        'Is_Promo': 1,                 # Promotional Campaign Active
        'Store_Footfall': 1650.0,      # Expected High Footfall
        'Avg_Discount_Pct': 20.0,      # 20% Average Discount
        'Seasonality_Factor': 1.25     # High Season/Holiday Demand
    }
    
    forecasted_sales = forecast_future_sales(model, upcoming_day)
    
    print("--- Upcoming Day Sales Forecasting Scenario ---")
    day_mapping = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}
    print(f"Day of Week             : {day_mapping[upcoming_day['Day_of_Week']]} ({upcoming_day['Day_of_Week']})")
    print(f"Promotional Campaign    : {'ACTIVE' if upcoming_day['Is_Promo'] == 1 else 'INACTIVE'}")
    print(f"Expected Store Footfall : {upcoming_day['Store_Footfall']:.0f} visitors")
    print(f"Average Discount        : {upcoming_day['Avg_Discount_Pct']}%")
    print(f"Seasonality Index       : {upcoming_day['Seasonality_Factor']}")
    
    print("-" * 60)
    print(f"Forecasted Sales Output : {forecasted_sales} Units")

--- Historical Daily Sales Dataset (First 5 Rows) ---
 Day_of_Week  Is_Promo  Store_Footfall  Avg_Discount_Pct  Seasonality_Factor  Units_Sold
           7         0         1867.86             26.06                1.36      2027.0
           4         0         1085.81             33.19                1.04      1345.0
           5         1          664.84              8.26                1.37      1465.0
           7         0         1026.44              0.44                0.86      1155.0
           3         0         1964.06              5.48                1.43      1440.0


--- Model Performance Metrics ---
R² Score (Accuracy Metric)     : 0.7648
Mean Absolute Error (MAE)      : 144.71 units
Root Mean Squared Error (RMSE) : 176.23 units


--- Key Sales Drivers (Feature Importance) ---
Feature [Day_of_Week         ] Impact: 10.25%
Feature [Is_Promo            ] Impact: 27.53%
Feature [Store_Footfall      ] Impact: 30.56%
Feature [Avg_Discount_Pct    ] Impact: 17.06%
Feature [Se